--- 
# **[실습]**

### 실습 목표

Contextual Retrieval을 실제 문서에 적용하여 검색 성능을 개선합니다.

### 난이도별 가이드

**기본 난이도:**
- 제공된 샘플 문서에 Contextual Retrieval 적용
- 일반 검색 vs Contextual 검색 성능 비교
- 3개 이상의 쿼리로 테스트

**중급 난이도:**
- 자체 문서(예: 기술 문서, 위키피디아) 활용
- Hybrid 검색 (Embedding + BM25) 구현
- 가중치 조합 실험 (0.3:0.7, 0.5:0.5, 0.7:0.3)

**고급 난이도:**
- 다국어 문서에 Contextual Retrieval 적용
- Reranker와 결합하여 최적 파이프라인 구성
- 검색 성능 지표 (HitRate, MRR) 측정 및 비교

`(1) 문서 준비 및 청킹`

자신의 문서를 로드하고 청크로 분할합니다.

In [7]:
# =============================================================================
# (1) 문서 준비 및 청킹 - 기본 난이도
# =============================================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from glob import glob
import os

# 문서 로드 (data 폴더의 한국어 마크다운 파일들)
txt_files = glob(os.path.join('data', '*_KR.md'))
print(f"로드할 파일: {txt_files}")

documents = []
for txt_file in txt_files:
    loader = TextLoader(txt_file, encoding='utf-8')
    documents.extend(loader.load())

print(f"로드된 문서 수: {len(documents)}")

# 전체 문서 내용 저장 (컨텍스트 생성에 사용)
whole_documents = {doc.metadata['source']: doc.page_content for doc in documents}

# 텍스트 분할기 설정 (300자 청크)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "],
)

# 문서 분할
all_chunks = text_splitter.split_documents(documents)

print(f"\n총 청크 수: {len(all_chunks)}")
print("="*80)

# 청크 예시 출력
for i, chunk in enumerate(all_chunks[:3]):
    print(f"\n[청크 {i+1}] (출처: {chunk.metadata['source']})")
    print("-"*40)
    print(chunk.page_content[:200] + "..." if len(chunk.page_content) > 200 else chunk.page_content)

로드할 파일: ['data\\Rivian_KR.md', 'data\\Tesla_KR.md']
로드된 문서 수: 2

총 청크 수: 61

[청크 1] (출처: data\Rivian_KR.md)
----------------------------------------
Rivian Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조업체, 자동차 기술 및 야외 레크리에이션 회사입니다.

**주요 정보:**

[청크 2] (출처: data\Rivian_KR.md)
----------------------------------------
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2...

[청크 3] (출처: data\Rivian_KR.md)
----------------------------------------
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러
- **총 자본 (2023):** 91억 4천만 미국 달러
- **직원 수 (2023년 12월):** 16,790명
- **웹사이트:** rivian.com


`(2) 컨텍스트 생성`

각 청크에 대해 맥락 설명을 생성합니다.

In [8]:
# =============================================================================
# (2) 컨텍스트 생성 - 중급 난이도
# =============================================================================

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi
from typing import List
from langfuse.langchain import CallbackHandler

# langfuse handler 설정 (이전 셀 실행 안 했을 경우 대비)
try:
    langfuse_handler
except NameError:
    langfuse_handler = CallbackHandler()

# LLM 초기화
context_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 컨텍스트 생성 프롬프트
context_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서의 청크에 맥락을 추가하는 전문가입니다.
주어진 청크가 전체 문서에서 어떤 위치에 있고 무엇에 대한 내용인지 간결하게 설명하세요.
설명은 50-100자 이내로 작성하세요. 오직 맥락 설명만 출력하세요."""),
    ("user", """<document>
{whole_document}
</document>

위 문서에서 아래 청크의 맥락을 설명해주세요:

<chunk>
{chunk_content}
</chunk>

맥락 설명:""")
])

context_chain = context_prompt | context_llm | StrOutputParser()

# 컨텍스트 생성 함수
def generate_contexts_for_chunks(chunks: List[Document], whole_docs: dict) -> List[str]:
    """청크들에 대한 컨텍스트를 배치로 생성합니다."""
    inputs = []
    for chunk in chunks:
        source = chunk.metadata.get('source', '')
        whole_doc = whole_docs.get(source, chunk.page_content)
        inputs.append({
            "whole_document": whole_doc[:3000],  # 토큰 제한 고려
            "chunk_content": chunk.page_content
        })
    
    contexts = context_chain.batch(
        inputs,
        config={"max_concurrency": 5, "callbacks": [langfuse_handler]}
    )
    return contexts

# 컨텍스트 생성 실행
print("컨텍스트 생성 중... (시간이 걸릴 수 있습니다)")
contexts = generate_contexts_for_chunks(all_chunks, whole_documents)

# Contextual 청크 생성
contextual_chunks = []
for i, (chunk, context) in enumerate(zip(all_chunks, contexts)):
    contextual_content = f"[맥락] {context}\n\n{chunk.page_content}"
    contextual_chunk = Document(
        page_content=contextual_content,
        metadata={
            **chunk.metadata,
            "chunk_id": i,
            "original_content": chunk.page_content,
            "context": context,
        }
    )
    contextual_chunks.append(contextual_chunk)

print(f"\nContextual 청크 생성 완료: {len(contextual_chunks)}개")
print("="*80)

# 예시 출력
for i in [0, 5, 10]:
    if i < len(contextual_chunks):
        print(f"\n[Contextual 청크 {i+1}]")
        print(f"맥락: {contextual_chunks[i].metadata['context']}")
        print(f"내용: {contextual_chunks[i].metadata['original_content'][:150]}...")

컨텍스트 생성 중... (시간이 걸릴 수 있습니다)

Contextual 청크 생성 완료: 61개

[Contextual 청크 1]
맥락: 이 청크는 Rivian의 설립 연도와 기본 개요를 소개하며, 문서의 서론 부분으로 회사의 배경과 핵심 정보를 제시하는 위치에 있습니다.
내용: Rivian Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조업체, 자동차 기술 및 야외 레크리에이션 회사입니다.

**주요 정보:**...

[Contextual 청크 6]
맥락: 이 청크는 Rivian의 생산 준비와 초기 제품 공개, 그리고 2021년 이후의 모델 배송, IPO, 인력 감축 등 확장 및 조정 과정을 다루며, 회사 성장의 핵심 전환기를 설명합니다.
내용: **생산 준비 (2016–20):**

- 2017년 일리노이 주 노멀에 있는 이전 Mitsubishi Motors 제조 공장을 1,600만 달러에 인수.
- 2017년 12월, 첫 두 제품인 R1T (픽업 트럭)와 R1S (SUV)를 공개.
- 생산은 2020년에 시...

[Contextual 청크 11]
맥락: 이 청크는 Rivian의 주요 차량 모델(R1T, R1S, EDV, R2, R3)과 향후 출시 예정인 차량에 대한 정보를 제공하며, 회사의 제품 라인업과 개발 계획을 설명하는 문서 내 차량 섹션에 위치합니다.
내용: **차량**

- **R1T:** 4개의 전기 모터가 장착된 픽업 트럭. 배터리 크기는 105 kWh에서 180 kWh까지 다양함.
- **R1S:** 첫 번째 Rivian 플랫폼의 스포츠 유틸리티 차량(SUV) 버전.
- **Electric Delivery Van (...


`(3) 하이브리드 검색 및 평가`

Embedding + BM25 + Reranker 파이프라인을 구성하고 성능을 평가합니다.

In [9]:
# =============================================================================
# (3) 하이브리드 검색 및 평가 - Gradio 데모
# =============================================================================

import gradio as gr
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
import numpy as np
import pandas as pd

# Kiwi 한국어 형태소 분석기 초기화
kiwi = Kiwi()
kiwi.add_user_word('리비안', 'NNP')
kiwi.add_user_word('테슬라', 'NNP')

def kiwi_preprocess_func(text):
    return [t.form for t in kiwi.tokenize(text)]

# 1. 벡터 저장소 생성
print("벡터 저장소 생성 중...")

# 일반 벡터 저장소
normal_vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="practice_normal_v2",
    persist_directory="./chroma_db"
)

# Contextual 벡터 저장소
contextual_vectorstore = Chroma.from_documents(
    documents=contextual_chunks,
    embedding=embeddings,
    collection_name="practice_contextual_v2",
    persist_directory="./chroma_db"
)

# 2. Retriever 생성
normal_retriever = normal_vectorstore.as_retriever(search_kwargs={"k": 5})
contextual_retriever = contextual_vectorstore.as_retriever(search_kwargs={"k": 5})

# BM25 Retriever
normal_bm25 = BM25Retriever.from_documents(all_chunks, preprocess_func=kiwi_preprocess_func, k=5)
contextual_bm25 = BM25Retriever.from_documents(contextual_chunks, preprocess_func=kiwi_preprocess_func, k=5)

# 3. 하이브리드 Retriever
normal_hybrid = EnsembleRetriever(retrievers=[normal_retriever, normal_bm25], weights=[0.5, 0.5])
contextual_hybrid = EnsembleRetriever(retrievers=[contextual_retriever, contextual_bm25], weights=[0.5, 0.5])

# 4. Reranker 설정
print("Reranker 설정 중...")
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

contextual_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=contextual_hybrid
)

# 모든 Retriever 딕셔너리
all_retrievers = {
    "일반 Embedding": normal_retriever,
    "일반 BM25": normal_bm25,
    "일반 Hybrid": normal_hybrid,
    "Contextual Embedding": contextual_retriever,
    "Contextual BM25": contextual_bm25,
    "Contextual Hybrid": contextual_hybrid,
    "Contextual + Reranker": contextual_rerank_retriever,
}

print("모든 Retriever 준비 완료!")

# =============================================================================
# Gradio 앱 함수들
# =============================================================================

def search_and_compare(query: str, top_k: int = 3):
    """쿼리에 대해 모든 Retriever로 검색하고 결과 비교"""
    results = {}
    
    for name, retriever in all_retrievers.items():
        try:
            docs = retriever.invoke(query)[:top_k]
            results[name] = []
            for i, doc in enumerate(docs):
                original = doc.metadata.get('original_content', doc.page_content)
                context = doc.metadata.get('context', '')
                results[name].append({
                    "rank": i + 1,
                    "content": original[:200] + "..." if len(original) > 200 else original,
                    "context": context,
                    "source": doc.metadata.get('source', 'N/A')
                })
        except Exception as e:
            results[name] = [{"rank": 1, "content": f"오류: {str(e)}", "context": "", "source": ""}]
    
    return results

def format_search_results(results: dict) -> str:
    """검색 결과를 마크다운 형식으로 포맷팅"""
    output = ""
    for retriever_name, docs in results.items():
        output += f"\n### 🔍 {retriever_name}\n"
        output += "-" * 50 + "\n"
        for doc in docs:
            output += f"**[{doc['rank']}]** {doc['content']}\n"
            if doc['context']:
                output += f"   *맥락: {doc['context']}*\n"
            output += "\n"
    return output

def evaluate_retrievers(test_queries_text: str):
    """테스트 쿼리로 모든 Retriever 성능 평가"""
    # 테스트 쿼리 파싱 (쿼리|키워드1,키워드2 형식)
    test_cases = []
    for line in test_queries_text.strip().split('\n'):
        if '|' in line:
            query, keywords = line.split('|', 1)
            test_cases.append((query.strip(), [k.strip() for k in keywords.split(',')]))
    
    if not test_cases:
        return "테스트 쿼리 형식이 올바르지 않습니다. '쿼리|키워드1,키워드2' 형식으로 입력하세요.", None, None
    
    # 평가 실행
    results = []
    for name, retriever in all_retrievers.items():
        hits = 0
        mrr_sum = 0
        
        for query, keywords in test_cases:
            try:
                docs = retriever.invoke(query)[:3]
                found = False
                for i, doc in enumerate(docs):
                    content = doc.metadata.get('original_content', doc.page_content).lower()
                    if any(kw.lower() in content for kw in keywords):
                        if not found:
                            hits += 1
                            mrr_sum += 1 / (i + 1)
                            found = True
                            break
            except:
                pass
        
        hit_rate = hits / len(test_cases) if test_cases else 0
        mrr = mrr_sum / len(test_cases) if test_cases else 0
        
        results.append({
            "Retriever": name,
            "Hit Rate": f"{hit_rate:.1%}",
            "MRR": f"{mrr:.3f}",
            "Hits": f"{hits}/{len(test_cases)}",
            "Category": "Contextual" if "Contextual" in name else "일반"
        })
    
    df = pd.DataFrame(results)
    
    # 성능 개선율 계산
    normal_avg = df[df['Category'] == '일반']['Hit Rate'].apply(lambda x: float(x.strip('%')) / 100).mean()
    contextual_avg = df[df['Category'] == 'Contextual']['Hit Rate'].apply(lambda x: float(x.strip('%')) / 100).mean()
    
    if normal_avg > 0:
        improvement = ((contextual_avg - normal_avg) / normal_avg) * 100
        summary = f"""
## 📊 성능 비교 요약

| 구분 | 평균 Hit Rate |
|------|--------------|
| 일반 검색 | {normal_avg:.1%} |
| Contextual 검색 | {contextual_avg:.1%} |

### 🚀 성능 개선율: **{improvement:+.1f}%**

{'✅ Contextual Retrieval이 일반 검색보다 우수합니다!' if improvement > 0 else '⚠️ 테스트 쿼리를 더 다양하게 구성해보세요.'}
"""
    else:
        summary = f"""
## 📊 성능 비교 요약

| 구분 | 평균 Hit Rate |
|------|--------------|
| 일반 검색 | {normal_avg:.1%} |
| Contextual 검색 | {contextual_avg:.1%} |

*일반 검색 Hit Rate가 0%입니다. 테스트 쿼리와 키워드를 확인해주세요.*
"""
    
    return summary, df

def gradio_search(query, top_k):
    """Gradio 검색 인터페이스"""
    if not query.strip():
        return "검색어를 입력하세요."
    results = search_and_compare(query, int(top_k))
    return format_search_results(results)

def gradio_evaluate(test_queries):
    """Gradio 평가 인터페이스"""
    summary, df = evaluate_retrievers(test_queries)
    return summary, df

# =============================================================================
# Gradio UI 구성
# =============================================================================

with gr.Blocks(title="Contextual Retrieval 성능 비교", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🔎 Contextual Retrieval 검색 성능 비교 데모
    
    **Contextual Retrieval**은 각 청크에 문서 전체 맥락을 추가하여 검색 성능을 향상시키는 기법입니다.
    이 데모에서 일반 검색과 Contextual 검색의 성능 차이를 직접 확인해보세요!
    """)
    
    with gr.Tabs():
        # 탭 1: 검색 비교
        with gr.Tab("🔍 실시간 검색 비교"):
            with gr.Row():
                query_input = gr.Textbox(
                    label="검색 쿼리",
                    placeholder="예: 리비안의 사업 경쟁력은 어디서 나오나요?",
                    lines=1
                )
                top_k_slider = gr.Slider(1, 5, value=3, step=1, label="검색 결과 수 (Top-K)")
            
            search_btn = gr.Button("🔍 검색 비교", variant="primary")
            search_output = gr.Markdown(label="검색 결과")
            
            search_btn.click(
                fn=gradio_search,
                inputs=[query_input, top_k_slider],
                outputs=search_output
            )
            
            gr.Examples(
                examples=[
                    ["리비안의 사업 경쟁력은 어디서 나오나요?", 3],
                    ["테슬라의 2023년 매출은 얼마인가요?", 3],
                    ["FSD 자율주행 기술 현황은?", 3],
                    ["4680 배터리 생산량은?", 3],
                ],
                inputs=[query_input, top_k_slider]
            )
        
        # 탭 2: 성능 평가
        with gr.Tab("📊 성능 평가"):
            gr.Markdown("""
            ### 테스트 쿼리 형식
            각 줄에 `쿼리|키워드1,키워드2,키워드3` 형식으로 입력하세요.
            키워드는 검색 결과에 포함되어야 할 단어들입니다.
            """)
            
            test_queries_input = gr.Textbox(
                label="테스트 쿼리 (줄바꿈으로 구분)",
                value="""리비안의 사업 경쟁력은?|리비안,경쟁,전기차
테슬라의 2023년 매출은?|매출,달러,967억
FSD 자율주행 기술 현황은?|FSD,자율주행,베타
4680 배터리 생산량은?|4680,배터리,셀
Model Y 판매량은?|Model Y,생산,120만""",
                lines=8
            )
            
            eval_btn = gr.Button("📊 성능 평가 실행", variant="primary")
            
            eval_summary = gr.Markdown(label="평가 요약")
            eval_table = gr.Dataframe(label="상세 결과")
            
            eval_btn.click(
                fn=gradio_evaluate,
                inputs=test_queries_input,
                outputs=[eval_summary, eval_table]
            )
        
        # 탭 3: 문서 정보
        with gr.Tab("📄 문서 정보"):
            gr.Markdown(f"""
            ### 로드된 문서 정보
            
            | 항목 | 값 |
            |------|-----|
            | 총 청크 수 | {len(all_chunks)} |
            | Contextual 청크 수 | {len(contextual_chunks)} |
            | 문서 출처 | {', '.join(set(c.metadata.get('source', 'N/A') for c in all_chunks))} |
            
            ### 청크 예시
            
            **[일반 청크]**
            ```
            {all_chunks[0].page_content[:300] if all_chunks else 'N/A'}...
            ```
            
            **[Contextual 청크]**
            ```
            {contextual_chunks[0].page_content[:400] if contextual_chunks else 'N/A'}...
            ```
            """)

print("Gradio 앱 준비 완료! 아래 셀에서 demo.launch()를 실행하세요.")

벡터 저장소 생성 중...
Reranker 설정 중...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 847.34it/s]


모든 Retriever 준비 완료!


C:\Users\sohee\AppData\Local\Temp\ipykernel_13404\17286662.py:208: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Contextual Retrieval 성능 비교", theme=gr.themes.Soft()) as demo:


Gradio 앱 준비 완료! 아래 셀에서 demo.launch()를 실행하세요.


In [15]:
# Gradio 앱 실행
# share=True로 설정하면 외부에서도 접속 가능한 공유 링크가 생성됩니다
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [14]:
demo.close()

Closing server running on port: 7860
